# Identificação de UCEs com PHYLUCE

Dataset: *Hypochilus petrunkevitchi* — SRR15736591  
Probe set: **Arachnida 1.1Kv1**

O desenho público Arachnida 1.1Kv1 contém 14.799 baits para 1.120 UCEs.

Nesta prática vamos:
- preparar o ambiente PHYLUCE;
- obter as sondas;
- organizar os contigs;
- procurar matches contig ↔ probe;
- gerar a lista de loci encontrados.

> Para a prática didática usamos SPAdes. O artigo original de *Hypochilus*
> usou Trinity + Velvet e matching a 80% de identidade/cobertura.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil, os

ROOT = Path("/content/drive/MyDrive/Bioinformatica_Biologia_Molecular")
ASSEMBLY = ROOT / "05_spades" / "SRR15736591" / "contigs_min200.fasta"
BASE = ROOT / "06_uce_match"
CONTIGS = BASE / "contigs"
PROBES = BASE / "probes"
RESULTS = BASE / "uce-search-results"

for p in [BASE, CONTIGS, PROBES, RESULTS]:
    p.mkdir(parents=True, exist_ok=True)

assert ASSEMBLY.exists(), "Execute primeiro o notebook de montagem."
target = CONTIGS / "SRR15736591.fasta"
shutil.copy2(ASSEMBLY, target)
print(target)

## 1. Preparar PHYLUCE com micromamba

A instalação pode levar alguns minutos na primeira execução.

In [ ]:
!mkdir -p /content/micromamba-bin
!wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content/micromamba-bin bin/micromamba
!/content/micromamba-bin/bin/micromamba create -y -p /content/phyluce-env -c conda-forge -c bioconda phyluce=1.7.3

## 2. Conferir PHYLUCE

In [ ]:
PHY = "/content/phyluce-env/bin"
!$PHY/phyluce_assembly_match_contigs_to_probes --help | head -20

## 3. Baixar o conjunto Arachnida 1.1Kv1

In [ ]:
import urllib.request, zipfile, tarfile, os, glob, shutil
from pathlib import Path

download = PROBES / "arachnida_1.1Kv1_download"
url = "https://ndownloader.figshare.com/files/6042078"
urllib.request.urlretrieve(url, download)
print("Baixado:", download, "bytes:", download.stat().st_size)

## 4. Identificar o formato baixado e localizar o FASTA de probes

O arquivo da Figshare pode ser distribuído como arquivo simples ou pacote.
A célula abaixo tenta descompactá-lo quando necessário e procura arquivos FASTA.

In [ ]:
import subprocess, os, shutil, tarfile, zipfile
from pathlib import Path

extract_dir = PROBES / "extracted"
extract_dir.mkdir(exist_ok=True)

# identificar pelo comando file
!file "$download"

# tentar ZIP
try:
    with zipfile.ZipFile(download) as z:
        z.extractall(extract_dir)
except Exception:
    pass

# tentar TAR
try:
    with tarfile.open(download) as t:
        t.extractall(extract_dir)
except Exception:
    pass

candidates = []
for pattern in ["*.fasta","*.fa","*.fas","*.fna"]:
    candidates.extend(extract_dir.rglob(pattern))

if not candidates:
    # pode ser FASTA sem extensão
    first = download.read_text(errors="ignore")[:1]
    if first == ">":
        probes_fasta = PROBES / "arachnida_1.1Kv1.fasta"
        shutil.copy2(download, probes_fasta)
    else:
        print("Arquivos encontrados:")
        !find "$PROBES" -maxdepth 3 -type f -printf "%p\n" | head -40
        raise FileNotFoundError("Não foi possível localizar automaticamente o FASTA de probes.")
else:
    # escolher o maior FASTA encontrado
    probes_fasta = max(candidates, key=lambda p: p.stat().st_size)

print("Probe FASTA:", probes_fasta)

## 5. Executar matching contigs ↔ probes

In [ ]:
probe_path = str(probes_fasta)
contigs_dir = str(CONTIGS)
results_dir = str(RESULTS)

!rm -rf "$results_dir"
!$PHY/phyluce_assembly_match_contigs_to_probes   --contigs "$contigs_dir"   --probes "$probe_path"   --output "$results_dir"   --min-coverage 80   --min-identity 80

## 6. Criar taxon-set e gerar contagem de matches

In [ ]:
taxon_conf = BASE / "taxon-set.conf"
taxon_conf.write_text("[all]\nSRR15736591\n")
print(taxon_conf.read_text())

In [ ]:
match_conf = BASE / "SRR15736591-incomplete.conf"
db = RESULTS / "probe.matches.sqlite"

!$PHY/phyluce_assembly_get_match_counts   --locus-db "$db"   --taxon-list-config "$taxon_conf"   --taxon-group all   --output "$match_conf"   --incomplete-matrix

## 7. Inspecionar o resultado

In [ ]:
print(match_conf.read_text()[:3000])

## Interpretação

O número de UCEs recuperados pode ser muito menor que no artigo porque utilizamos
uma fração dos reads. Se necessário, repita as aulas anteriores com `MAX_SPOTS`
maior e compare a recuperação.